# Task 1: Fine-Tuning GPT-2 for Python Code Generation

**Student:** Furkhanuddin Mohammad (ID: 16667990)

**Module:** 7043SCN - Generative AI and Reinforcement Learning

**Model:** GPT-2 (124M params, decoder-only autoregressive)

**Dataset:** iamtarun/python_code_instructions_18k_alpaca

**Technique:** LoRA (Low-Rank Adaptation)

In [1]:
import subprocess, sys
for pkg in ["torch", "transformers>=4.36.0", "datasets>=2.16.0", "peft>=0.7.0",
            "accelerate>=0.25.0", "evaluate>=0.4.0", "rouge-score", "nltk",
            "scikit-learn", "matplotlib", "seaborn", "sacrebleu"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages installed.")

All packages installed.


In [2]:
import os, time, json, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import torch
import nltk
from datasets import load_dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer,
                          TrainingArguments, DataCollatorForLanguageModeling, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CONFIG = {
    "model_name": "gpt2",
    "dataset_name": "iamtarun/python_code_instructions_18k_alpaca",
    "max_length": 256,
    "batch_size": 4, "learning_rate": 5e-4, "num_epochs": 3,
    "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.1,
    "output_dir": "./results", "seed": SEED,
    "num_samples": 5000,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [3]:
dataset = load_dataset(CONFIG["dataset_name"])
print(f"Dataset: {len(dataset['train'])} samples")
print(f"Columns: {dataset['train'].column_names}")
print(f"Sample: {dataset['train'][0]}")

full = dataset["train"].shuffle(seed=SEED).select(range(min(CONFIG["num_samples"], len(dataset["train"]))))
n = len(full)
split_dataset = DatasetDict({
    "train": full.select(range(int(n*0.8))),
    "validation": full.select(range(int(n*0.8), int(n*0.9))),
    "test": full.select(range(int(n*0.9), n)),
})
print(f"Train: {len(split_dataset['train'])}, Val: {len(split_dataset['validation'])}, Test: {len(split_dataset['test'])}")

Dataset: 18612 samples
Columns: ['instruction', 'input', 'output', 'prompt']
Sample: {'instruction': 'Create a function to calculate the sum of a sequence of integers.', 'input': '[1, 2, 3, 4, 5]', 'output': '# Python code\ndef sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum', 'prompt': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nCreate a function to calculate the sum of a sequence of integers.\n\n### Input:\n[1, 2, 3, 4, 5]\n\n### Output:\n# Python code\ndef sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum'}
Train: 4000, Val: 500, Test: 500


In [4]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
tokenizer.pad_token = tokenizer.eos_token

def format_sample(example):
    prompt = example.get("prompt", example.get("instruction", ""))
    inp = example.get("input", "")
    output = example.get("output", "")
    if inp and inp.strip():
        text = f"### Instruction:\n{prompt}\n### Input:\n{inp}\n### Output:\n{output}"
    else:
        text = f"### Instruction:\n{prompt}\n### Output:\n{output}"
    return text

def tokenize_fn(examples):
    texts = []
    for i in range(len(examples[list(examples.keys())[0]])):
        sample = {k: examples[k][i] for k in examples}
        texts.append(format_sample(sample))
    tokenized = tokenizer(texts, max_length=CONFIG["max_length"], truncation=True, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized = split_dataset.map(tokenize_fn, batched=True, remove_columns=split_dataset["train"].column_names)
print("Tokenization complete.")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization complete.


## Baseline Evaluation (Zero-Shot)

In [5]:
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

baseline_model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"]).to(device)
baseline_model.eval()
model_params = sum(p.numel() for p in baseline_model.parameters())
print(f"Model parameters: {model_params:,} ({model_params/1e6:.1f}M)")

test_samples = list(split_dataset["test"])
num_eval = min(100, len(test_samples))
baseline_preds, refs, prompts = [], [], []
t0 = time.time()
with torch.no_grad():
    for i in range(num_eval):
        sample = test_samples[i]
        prompt = sample.get("prompt", sample.get("instruction", ""))
        inp = sample.get("input", "")
        ref = sample.get("output", "")
        if inp and inp.strip():
            input_text = f"### Instruction:\n{prompt}\n### Input:\n{inp}\n### Output:\n"
        else:
            input_text = f"### Instruction:\n{prompt}\n### Output:\n"
        enc = tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True).to(device)
        out = baseline_model.generate(**enc, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        baseline_preds.append(gen if gen else "empty")
        refs.append(ref.strip() if ref.strip() else "empty")
        prompts.append(prompt)
        if (i+1) % 25 == 0: print(f"  {i+1}/{num_eval}")
baseline_eval_time = time.time() - t0

bl_rouge = rouge_metric.compute(predictions=baseline_preds, references=refs, use_stemmer=True)
bl_bleu = []
for p, r in zip(baseline_preds, refs):
    try:
        pt, rt = p.split(), r.split()
        bl_bleu.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"] if pt and rt else 0.0)
    except: bl_bleu.append(0.0)

baseline_metrics = {"rouge1": round(bl_rouge["rouge1"],4), "rouge2": round(bl_rouge["rouge2"],4),
                    "rougeL": round(bl_rouge["rougeL"],4), "bleu": round(np.mean(bl_bleu),4)}
print(f"Baseline: R1={baseline_metrics['rouge1']}, R2={baseline_metrics['rouge2']}, RL={baseline_metrics['rougeL']}, BLEU={baseline_metrics['bleu']}")

for i in range(min(3, num_eval)):
    print(f"\nPrompt: {prompts[i][:120]}")
    print(f"Pred: {baseline_preds[i][:120]}")
    print(f"Ref: {refs[i][:120]}")

del baseline_model
if torch.cuda.is_available(): torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model parameters: 124,439,808 (124.4M)


  25/100


  50/100


  75/100


## Hardware Assessment

Full fine-tuning of GPT-2 (124M parameters) requires storing all parameter gradients and optimizer states, demanding approximately 4-6 GB of GPU memory. On CPU-only hardware, full fine-tuning is **infeasible** due to excessive training time (estimated 15+ hours). We use **LoRA (Low-Rank Adaptation)** which adds small trainable rank-decomposition matrices to attention layers, reducing trainable parameters by ~97% while preserving model capacity.

## LoRA Fine-Tuning

In [ ]:
model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"])
model.config.pad_token_id = tokenizer.eos_token_id

lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=CONFIG["lora_r"],
                         lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
                         target_modules=["c_attn", "c_proj"], bias="none")
model = get_peft_model(model, lora_config)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
reduction = (1 - trainable/total_params) * 100
print(f"LoRA: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, targets=[c_attn, c_proj]")
print(f"Total: {total_params:,}, Trainable: {trainable:,} ({trainable/total_params*100:.4f}%), Reduction: {reduction:.2f}%")

In [ ]:
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"], num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"], per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"], weight_decay=0.01, warmup_steps=100,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=50, report_to="none", seed=SEED, fp16=torch.cuda.is_available(), dataloader_num_workers=0,
    optim="adamw_torch",
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Training: {CONFIG['num_epochs']} epochs, batch={CONFIG['batch_size']}, lr={CONFIG['learning_rate']}")
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0
print(f"Done in {train_time:.1f}s ({train_time/60:.1f} min), loss={train_result.training_loss:.4f}")

epoch_metrics = [e for e in trainer.state.log_history if "eval_loss" in e and "epoch" in e]
for em in epoch_metrics:
    print(f"  Epoch {em['epoch']:.0f}: Loss={em['eval_loss']:.4f}")

## LoRA Evaluation on Test Set

In [ ]:
lora_preds = []
model.eval()
t0 = time.time()
with torch.no_grad():
    for i in range(num_eval):
        sample = test_samples[i]
        prompt = sample.get("prompt", sample.get("instruction", ""))
        inp = sample.get("input", "")
        if inp and inp.strip():
            input_text = f"### Instruction:\n{prompt}\n### Input:\n{inp}\n### Output:\n"
        else:
            input_text = f"### Instruction:\n{prompt}\n### Output:\n"
        enc = tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True).to(device)
        out = model.generate(**enc, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        lora_preds.append(gen if gen else "empty")
        if (i+1) % 25 == 0: print(f"  {i+1}/{num_eval}")
lora_eval_time = time.time() - t0

lora_rouge = rouge_metric.compute(predictions=lora_preds, references=refs, use_stemmer=True)
lora_bleu = []
for p, r in zip(lora_preds, refs):
    try:
        pt, rt = p.split(), r.split()
        lora_bleu.append(bleu_metric.compute(predictions=[pt], references=[[rt]])["bleu"] if pt and rt else 0.0)
    except: lora_bleu.append(0.0)

lora_metrics = {"rouge1": round(lora_rouge["rouge1"],4), "rouge2": round(lora_rouge["rouge2"],4),
                "rougeL": round(lora_rouge["rougeL"],4), "bleu": round(np.mean(lora_bleu),4)}
print(f"LoRA: R1={lora_metrics['rouge1']}, R2={lora_metrics['rouge2']}, RL={lora_metrics['rougeL']}, BLEU={lora_metrics['bleu']}")

for i in range(min(3, num_eval)):
    print(f"\nPrompt: {prompts[i][:120]}")
    print(f"LoRA: {lora_preds[i][:120]}")
    print(f"Ref: {refs[i][:120]}")

## Comparative Analysis and Visualization

In [ ]:
import pandas as pd
comparison = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"],
    "Baseline": [baseline_metrics["rouge1"], baseline_metrics["rouge2"], baseline_metrics["rougeL"], baseline_metrics["bleu"]],
    "LoRA": [lora_metrics["rouge1"], lora_metrics["rouge2"], lora_metrics["rougeL"], lora_metrics["bleu"]],
})
comparison["Improvement"] = comparison["LoRA"] - comparison["Baseline"]
comparison["Improvement_%"] = ((comparison["Improvement"] / comparison["Baseline"].replace(0,1)) * 100).round(2)
print(comparison.to_string(index=False))
print(f"\nTotal params: {total_params:,}, Trainable: {trainable:,}, Reduction: {reduction:.2f}%")
print(f"Training: {train_time:.1f}s, Baseline eval: {baseline_eval_time:.1f}s, LoRA eval: {lora_eval_time:.1f}s")

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
metrics_names = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
bl_vals = [baseline_metrics["rouge1"], baseline_metrics["rouge2"], baseline_metrics["rougeL"], baseline_metrics["bleu"]]
lo_vals = [lora_metrics["rouge1"], lora_metrics["rouge2"], lora_metrics["rougeL"], lora_metrics["bleu"]]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

x = np.arange(len(metrics_names)); w = 0.35
axes[0,0].bar(x-w/2, bl_vals, w, label="Baseline", color="#3498db", edgecolor="black", linewidth=0.5)
axes[0,0].bar(x+w/2, lo_vals, w, label="LoRA", color="#e74c3c", edgecolor="black", linewidth=0.5)
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(metrics_names)
axes[0,0].set_title("Baseline vs LoRA"); axes[0,0].legend(); axes[0,0].grid(axis="y", alpha=0.3)

if epoch_metrics:
    eps = [e["epoch"] for e in epoch_metrics]
    axes[0,1].plot(eps, [e["eval_loss"] for e in epoch_metrics], "b-o", lw=2)
    axes[0,1].set_title("Validation Loss"); axes[0,1].set_xlabel("Epoch"); axes[0,1].grid(alpha=0.3)

axes[1,0].bar(["Total", "Trainable (LoRA)"], [total_params, trainable], color=["#3498db", "#e74c3c"], edgecolor="black")
axes[1,0].set_title("Parameter Efficiency"); axes[1,0].set_yscale("log"); axes[1,0].grid(axis="y", alpha=0.3)

N = len(metrics_names)
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]
ax = axes[1,1]
ax.remove()
ax = fig.add_subplot(2, 2, 4, polar=True)
ax.plot(angles, bl_vals + bl_vals[:1], "o-", lw=2, label="Baseline", color="#3498db")
ax.fill(angles, bl_vals + bl_vals[:1], alpha=0.15, color="#3498db")
ax.plot(angles, lo_vals + lo_vals[:1], "o-", lw=2, label="LoRA", color="#e74c3c")
ax.fill(angles, lo_vals + lo_vals[:1], alpha=0.15, color="#e74c3c")
ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics_names)
ax.set_title("Radar"); ax.legend(loc="upper right", fontsize=8)

plt.suptitle("GPT-2 Code Generation: LoRA Fine-Tuning Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/results.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
import seaborn as sns

# ============================================================
# Figure 1: Improvement Analysis (Absolute + Percentage)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_vals = [lo - bl for lo, bl in zip(lo_vals, bl_vals)]
colors_imp = ["#2ecc71" if v >= 0 else "#e74c3c" for v in imp_vals]
axes[0].bar(metrics_names, imp_vals, color=colors_imp, edgecolor="black", linewidth=0.5)
axes[0].axhline(y=0, color="black", linewidth=0.8, linestyle="--")
for i, v in enumerate(imp_vals):
    axes[0].text(i, v + (0.005 if v >= 0 else -0.02), f"{v:+.4f}", ha="center", fontsize=9, fontweight="bold")
axes[0].set_title("Absolute Improvement (LoRA - Baseline)", fontsize=11)
axes[0].set_ylabel("Score Difference"); axes[0].grid(axis="y", alpha=0.3)

pct_vals = [(lo - bl) / bl * 100 if bl != 0 else 0 for lo, bl in zip(lo_vals, bl_vals)]
colors_pct = ["#2ecc71" if v >= 0 else "#e74c3c" for v in pct_vals]
axes[1].bar(metrics_names, pct_vals, color=colors_pct, edgecolor="black", linewidth=0.5)
axes[1].axhline(y=0, color="black", linewidth=0.8, linestyle="--")
for i, v in enumerate(pct_vals):
    axes[1].text(i, v + (2 if v >= 0 else -5), f"{v:+.1f}%", ha="center", fontsize=9, fontweight="bold")
axes[1].set_title("Percentage Improvement", fontsize=11)
axes[1].set_ylabel("Improvement (%)"); axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("GPT-2 Code Generation: Improvement Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/improvement_analysis.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 2: Training Dynamics (Step-Level Loss + Epoch Loss)
# ============================================================
train_steps = [e for e in trainer.state.log_history if "loss" in e and "eval_loss" not in e]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_steps:
    steps = [e["step"] for e in train_steps]
    losses = [e["loss"] for e in train_steps]
    axes[0].plot(steps, losses, color="#e74c3c", linewidth=1.0, alpha=0.5, label="Raw Loss")
    if len(losses) > 5:
        w = min(10, max(3, len(losses)//3))
        ma = np.convolve(losses, np.ones(w)/w, mode="valid")
        axes[0].plot(steps[w-1:], ma, color="#2c3e50", linewidth=2.5, label=f"Moving Avg (w={w})")
    axes[0].set_title("Training Loss per Step"); axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(alpha=0.3)

if epoch_metrics:
    eps = [e["epoch"] for e in epoch_metrics]
    axes[1].plot(eps, [e["eval_loss"] for e in epoch_metrics], "b-o", lw=2, markersize=8, label="Eval Loss")
    for i, (ep, loss) in enumerate(zip(eps, [e["eval_loss"] for e in epoch_metrics])):
        axes[1].annotate(f"{loss:.4f}", (ep, loss), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
    axes[1].set_title("Validation Loss Across Epochs"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("GPT-2 Code Generation: Training Dynamics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/training_dynamics.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 3: Prediction Analysis (Length Distribution + BLEU Box)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bl_lens = [len(p.split()) for p in baseline_preds]
lo_lens = [len(p.split()) for p in lora_preds]
ref_lens = [len(r.split()) for r in refs]

axes[0].hist(ref_lens, bins=20, alpha=0.4, label=f"Reference (mean={np.mean(ref_lens):.1f})", color="#2ecc71", edgecolor="black")
axes[0].hist(bl_lens, bins=20, alpha=0.5, label=f"Baseline (mean={np.mean(bl_lens):.1f})", color="#3498db", edgecolor="black")
axes[0].hist(lo_lens, bins=20, alpha=0.5, label=f"LoRA (mean={np.mean(lo_lens):.1f})", color="#e74c3c", edgecolor="black")
axes[0].set_title("Prediction Length Distribution"); axes[0].set_xlabel("Words"); axes[0].set_ylabel("Frequency")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

bp = axes[1].boxplot([bl_bleu, lora_bleu], labels=["Baseline", "LoRA"], patch_artist=True,
                      medianprops=dict(color="black", linewidth=2))
bp["boxes"][0].set_facecolor("#3498db"); bp["boxes"][0].set_alpha(0.6)
bp["boxes"][1].set_facecolor("#e74c3c"); bp["boxes"][1].set_alpha(0.6)
axes[1].set_title("Per-Sample BLEU Distribution"); axes[1].set_ylabel("BLEU Score"); axes[1].grid(alpha=0.3)

plt.suptitle("GPT-2 Code Generation: Prediction Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/prediction_analysis.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 4: Summary Dashboard (Radar + Pie + Timing + Heatmap)
# ============================================================
fig = plt.figure(figsize=(16, 12))

ax1 = fig.add_subplot(2, 2, 1, polar=True)
N = len(metrics_names)
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]
ax1.plot(angles, bl_vals + bl_vals[:1], "o-", lw=2, label="Baseline", color="#3498db")
ax1.fill(angles, bl_vals + bl_vals[:1], alpha=0.15, color="#3498db")
ax1.plot(angles, lo_vals + lo_vals[:1], "o-", lw=2, label="LoRA", color="#e74c3c")
ax1.fill(angles, lo_vals + lo_vals[:1], alpha=0.15, color="#e74c3c")
ax1.set_xticks(angles[:-1]); ax1.set_xticklabels(metrics_names, fontsize=9)
ax1.set_title("Performance Radar", fontsize=12, pad=20)
ax1.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)

ax2 = fig.add_subplot(2, 2, 2)
frozen = total_params - trainable
ax2.pie([trainable, frozen], labels=[f"Trainable\n({trainable:,})", f"Frozen\n({frozen:,})"],
        colors=["#e74c3c", "#3498db"], autopct="%1.2f%%", startangle=90,
        textprops={"fontsize": 9}, explode=[0.05, 0])
ax2.set_title("LoRA Parameter Distribution", fontsize=12)

ax3 = fig.add_subplot(2, 2, 3)
times = [baseline_eval_time, train_time, lora_eval_time]
time_labels = ["Baseline\nEval", "LoRA\nTraining", "LoRA\nEval"]
bars = ax3.bar(time_labels, times, color=["#3498db", "#e74c3c", "#2ecc71"], edgecolor="black", linewidth=0.5)
for bar, t in zip(bars, times):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(times)*0.02,
             f"{t:.1f}s", ha="center", fontsize=10, fontweight="bold")
ax3.set_title("Timing Comparison"); ax3.set_ylabel("Time (seconds)"); ax3.grid(axis="y", alpha=0.3)

ax4 = fig.add_subplot(2, 2, 4)
heatmap_data = np.array([bl_vals, lo_vals])
sns.heatmap(heatmap_data, annot=True, fmt=".4f", xticklabels=metrics_names,
            yticklabels=["Baseline", "LoRA"], cmap="YlOrRd", ax=ax4,
            linewidths=0.5, linecolor="black", cbar_kws={"shrink": 0.8})
ax4.set_title("Metrics Heatmap", fontsize=12)

plt.suptitle("GPT-2 Code Generation: Summary Dashboard", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/dashboard.png", dpi=200, bbox_inches="tight")
plt.show()

# ============================================================
# Figure 5: BLEU Score Deep Analysis (CDF + Scatter)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sorted_bl = np.sort(bl_bleu)
sorted_lo = np.sort(lora_bleu)
axes[0].plot(sorted_bl, np.linspace(0, 1, len(sorted_bl)), lw=2, label="Baseline", color="#3498db")
axes[0].plot(sorted_lo, np.linspace(0, 1, len(sorted_lo)), lw=2, label="LoRA", color="#e74c3c")
axes[0].set_title("Cumulative BLEU Distribution (CDF)"); axes[0].set_xlabel("BLEU Score")
axes[0].set_ylabel("Cumulative Proportion"); axes[0].legend(); axes[0].grid(alpha=0.3)

min_len = min(len(bl_bleu), len(lora_bleu))
axes[1].scatter(bl_bleu[:min_len], lora_bleu[:min_len], alpha=0.4, s=20, color="#8e44ad")
max_v = max(max(bl_bleu[:min_len]) if bl_bleu[:min_len] else 0.01, max(lora_bleu[:min_len]) if lora_bleu[:min_len] else 0.01, 0.01)
axes[1].plot([0, max_v], [0, max_v], "k--", lw=1.5, label="y=x (equal performance)")
axes[1].set_title("Per-Sample BLEU: Baseline vs LoRA"); axes[1].set_xlabel("Baseline BLEU")
axes[1].set_ylabel("LoRA BLEU"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("GPT-2 Code Generation: BLEU Score Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/bleu_analysis.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"All extended visualizations saved to {CONFIG['output_dir']}/")

In [ ]:
results = {
    "student": "Furkhanuddin Mohammad (16667990)", "config": CONFIG,
    "baseline_metrics": baseline_metrics, "lora_metrics": lora_metrics,
    "training_time_seconds": train_time, "total_parameters": total_params,
    "trainable_parameters": trainable, "parameter_reduction_percent": round(reduction, 2),
    "epoch_metrics": epoch_metrics,
}
with open(f"{CONFIG['output_dir']}/results.json", "w") as f:
    json.dump(results, f, indent=2)
comparison.to_csv(f"{CONFIG['output_dir']}/comparison.csv", index=False)
model.save_pretrained(f"{CONFIG['output_dir']}/lora_model")
tokenizer.save_pretrained(f"{CONFIG['output_dir']}/lora_model")
print(f"All results saved to {CONFIG['output_dir']}/")